# OLS Signal Trading

This will be a stand in repo for testing the signal trading

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global Variables and Data

In [2]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path, ".."))
src_path  = os.path.join(repo_path, "src")
data_path = os.path.join(repo_path, "data")

In [3]:
sys.path.append(src_path)
from SignalFitter import SignalFitter

In [4]:
trend_path = os.path.join(data_path, "GenericTrendSignal.parquet")
df_data    = (pd
    .read_parquet(path = trend_path, engine = "pyarrow")
    .assign(signal_name = lambda x: x.Ticker + "_" + x.window.astype(str)))

In [5]:
print(src_path)

A:\GitHub\SignalTrader\src


# OLS Models

In [ ]:
signal_fitter = SignalFitter(rtn_col = "rtn", group_col = "signal_name", date_col = "Date")

In [ ]:
lag_full_sample        = signal_fitter.fit_full_sample(df_data, signal_col = "lag_signal")
lag_full_sample_params = signal_fitter.get_params(lag_full_sample)

In [ ]:
full_sample        = signal_fitter.fit_full_sample(df_data, signal_col = "signal")
full_sample_params = signal_fitter.get_params(full_sample)

In [ ]:
df_nolag = (full_sample_params
    .loc[lambda x: x.variable == "signal"]
    [["signal_name", "params"]]
    .rename(columns = {"params": "no_lag"}))

In [ ]:
df_lag = (lag_full_sample_params
    .loc[lambda x: x.variable == "lag_signal"]
    [["signal_name", "params"]]
    .rename(columns = {"params": "lag"}))

In [ ]:
cutoff_date    = "2018-12-31"
lag_train_test = signal_fitter.fit_train_test_date(df_data, signal_col = "lag_signal", cutoff_date = cutoff_date)
lag_expanding  = signal_fitter.fit_expanding(df_data, signal_col = "lag_signal")

In [ ]:
print(lag_full_sample)
print(lag_train_test)
print(lag_expanding)

In [ ]:
lag_train_test["SPY_64"].summary()

In [ ]:
lag_expanding["SPY_64"].tail()

# Fitted Value Backtests

Fit on the lagged signal, predicted_t = const + beta * lag_signal_t, position_t = sign(predicted_t) earning rtn_t. No vol targeting

In [ ]:
bt_fitted_full       = signal_fitter.backtest_fitted(lag_full_sample)
bt_fitted_train_test = signal_fitter.backtest_fitted(lag_train_test)
bt_fitted_expanding  = signal_fitter.backtest_fitted(lag_expanding)

In [ ]:
df_fitted_stats = (pd.concat([
    signal_fitter.get_backtest_stats(lag_full_sample, bt_fitted_full),
    signal_fitter.get_backtest_stats(lag_train_test, bt_fitted_train_test),
    signal_fitter.get_backtest_stats(lag_expanding, bt_fitted_expanding)]))

In [ ]:
(df_fitted_stats
    .groupby("sample")
    [["sharpe", "hit_rate", "pct_active", "r2", "corr"]]
    .mean())

# Residual Backtests

Same day fit on the unlagged signal, resid_t = y_t - (const + beta * signal_t), position_t+1 = -sign(resid_t) earning rtn_t+1. No vol targeting

In [ ]:
train_test = signal_fitter.fit_train_test_date(df_data, signal_col = "signal", cutoff_date = cutoff_date)
expanding  = signal_fitter.fit_expanding(df_data, signal_col = "signal")

In [ ]:
bt_resid_full       = signal_fitter.backtest_residual(full_sample)
bt_resid_train_test = signal_fitter.backtest_residual(train_test)
bt_resid_expanding  = signal_fitter.backtest_residual(expanding)

In [ ]:
df_resid_stats = (pd.concat([
    signal_fitter.get_backtest_stats(full_sample, bt_resid_full),
    signal_fitter.get_backtest_stats(train_test, bt_resid_train_test),
    signal_fitter.get_backtest_stats(expanding, bt_resid_expanding)]))

In [ ]:
(df_resid_stats
    .groupby("sample")
    [["sharpe", "hit_rate", "pct_active"]]
    .mean())

In [ ]:
(pd.concat([
    bt_fitted_expanding.assign(strategy = "fitted"),
    bt_resid_expanding.assign(strategy = "residual")])
    .groupby(["Date", "strategy"])
    .signal_rtn
    .mean()
    .unstack()
    .cumsum()
    .plot(title = "Expanding, equal weight across signal_name (no vol targeting)"))

# Checks

Checks on the real data, each prints PASS / FAIL. The last cell runs the synthetic test suite in SignalFitter.main()

In [ ]:
import statsmodels.api as sm
import SignalFitter as signal_fitter_module

check_name = "SPY_64"

def check(name: str, condition: bool) -> None:
    print("{:<65} {}".format(name, "PASS" if condition else "FAIL"))

In [ ]:
check("FitResult: fit_result[group] is a statsmodels model", hasattr(lag_full_sample[check_name], "params"))
check("FitResult: config records the signal and target", 
    (lag_full_sample.config["signal_col"] == "lag_signal") and (lag_full_sample.config["y_col"] == "vol_norm_rtn"))
check("FitResult: no predictions stored in the fit", "predicted" not in lag_full_sample.data.columns)
check("FitResult: one object per signal_name", len(lag_full_sample) + len(lag_full_sample.skipped) == df_data.signal_name.nunique())

In [ ]:
bt_tmp = bt_fitted_full.loc[lambda x: x.signal_name == check_name]

check("Fitted in-sample: predicted equals statsmodels fittedvalues", np.allclose(
    bt_tmp.predicted.values, lag_full_sample[check_name].fittedvalues.values))
check("Fitted in-sample: position = sign(predicted)", np.array_equal(
    bt_fitted_full.position.values, np.sign(bt_fitted_full.predicted.values)))

In [ ]:
df_train = (lag_train_test.data
    .loc[lambda x: (x.signal_name == check_name) & (x.Date <= lag_train_test.splits[check_name]["train_end"])])

train_model = sm.OLS(df_train.vol_norm_rtn, sm.add_constant(df_train.lag_signal, has_constant = "add")).fit()
bt_tmp      = bt_fitted_train_test.loc[lambda x: (x.signal_name == check_name) & (x["sample"] == "test")]

check("Train/test: model equals sm.OLS on train rows only", np.allclose(
    train_model.params.values, lag_train_test[check_name].params.values))
check("Train/test: train_end <= cutoff < test_start for every signal", all(
    (split["train_end"] <= pd.Timestamp(cutoff_date)) and (split["test_start"] > pd.Timestamp(cutoff_date))
    for split in lag_train_test.splits.values()))
check("Train/test: test predicted uses the train params", np.allclose(
    bt_tmp.predicted.values,
    train_model.params.const + train_model.params.lag_signal * bt_tmp.lag_signal.values))

In [ ]:
df_exp_data = lag_expanding.data.loc[lambda x: x.signal_name == check_name].reset_index(drop = True)
point       = 1_000
df_hist     = df_exp_data.iloc[:point]
hist_model  = sm.OLS(df_hist.vol_norm_rtn, sm.add_constant(df_hist.lag_signal, has_constant = "add")).fit()
bt_tmp      = bt_fitted_expanding.loc[lambda x: x.signal_name == check_name].set_index("Date")

check("Expanding: predicted_t uses params fit through t-1", np.isclose(
    bt_tmp.loc[df_exp_data.Date.iloc[point], "predicted"],
    hist_model.params.const + hist_model.params.lag_signal * df_exp_data.lag_signal.iloc[point]))
check("Expanding: last params equal the full sample fit", np.allclose(
    lag_expanding[check_name][["const_param", "lag_signal_param"]].iloc[-1].values,
    lag_full_sample[check_name].params.values))

In [ ]:
lag_one_fit = signal_fitter.fit_full_sample(df_data, signal_col = "signal", lag = 1)

try:
    signal_fitter.backtest_residual(lag_one_fit)
    raised = False
except ValueError:
    raised = True

check("Residual: refuses a fit made with lag = 1", raised)

In [ ]:
df_tmp = full_sample.data.loc[lambda x: x.signal_name == check_name]
resid  = full_sample[check_name].resid.values
bt_tmp = bt_resid_full.loc[lambda x: x.signal_name == check_name]

check("Residual in-sample: resid equals statsmodels resid", 
    (len(bt_tmp) == len(resid) - 1) and np.allclose(bt_tmp.resid.values, resid[1:]))
check("Residual: position_t = -sign(resid_t-1)", 
    (len(bt_tmp) == len(resid) - 1) and np.array_equal(bt_tmp.position.values, -np.sign(resid[:-1])))
check("Residual: first day of every signal_name has no position", 
    len(bt_resid_full) == len(full_sample.data) - len(full_sample))

In [ ]:
df_exp_data = expanding.data.loc[lambda x: x.signal_name == check_name].reset_index(drop = True)
point       = 1_000
df_hist     = df_exp_data.iloc[:point]
hist_model  = sm.OLS(df_hist.vol_norm_rtn, sm.add_constant(df_hist.signal, has_constant = "add")).fit()
resid_point = df_exp_data.vol_norm_rtn.iloc[point] - (hist_model.params.const + hist_model.params.signal * df_exp_data.signal.iloc[point])
bt_tmp      = bt_resid_expanding.loc[lambda x: x.signal_name == check_name].set_index("Date")

check("Residual expanding: resid_t uses params fit through t-1", np.isclose(
    bt_tmp.loc[df_exp_data.Date.iloc[point], "resid"], resid_point))
check("Residual expanding: traded the next day as -sign(resid_t)",
    bt_tmp.loc[df_exp_data.Date.iloc[point + 1], "position"] == -np.sign(resid_point))

In [ ]:
signal_fitter_module.main()